In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import sounddevice as sd
import scipy.io.wavfile as wavfile
import numpy as np
import whisper
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch
import pyttsx3
import os
import threading
import time

# Initialize models and components
# 1. Speech-to-Text (Whisper-small)
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")

# 2. Image Question-Answering (BLIP-2 ViT-base + Flan-T5-small)
print("Loading BLIP-2 model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-base")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-base",
    torch_dtype=torch.float16
).to(device)

# 3. Text-to-Speech (pyttsx3)
print("Initializing TTS engine...")
tts_engine = pyttsx3.init()
tts_engine.setProperty('rate', 150)  # Speed of speech

# Global variables
SAMPLE_RATE = 16000  # Whisper expects 16kHz audio
RECORD_SECONDS = 10
audio_file = "temp_audio.wav"
image_path = None
is_recording = False

# GUI Class
class VQAInterface:
    def __init__(self, root):
        self.root = root
        self.root.title("Speech-to-Text Image QA System")
        self.root.geometry("600x500")

        # GUI Elements
        # Image display
        self.image_label = tk.Label(self.root, text="No image uploaded", width=40, height=10)
        self.image_label.pack(pady=10)

        # Upload Image Button
        self.upload_btn = tk.Button(self.root, text="Upload Image", command=self.upload_image)
        self.upload_btn.pack(pady=5)

        # Record Button
        self.record_btn = tk.Button(self.root, text="Record Question (10s)", command=self.start_recording)
        self.record_btn.pack(pady=5)

        # Question Display
        self.question_label = tk.Label(self.root, text="Question: (Waiting for speech input)", wraplength=500)
        self.question_label.pack(pady=5)

        # Answer Display
        self.answer_label = tk.Label(self.root, text="Answer: (Waiting for question)", wraplength=500)
        self.answer_label.pack(pady=5)

        # Play Answer Button
        self.play_btn = tk.Button(self.root, text="Play Answer", command=self.play_answer, state=tk.DISABLED)
        self.play_btn.pack(pady=5)

        # Current answer for TTS
        self.current_answer = ""

    def upload_image(self):
        global image_path
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.png *.jpg *.jpeg *.bmp *.gif")])
        if file_path:
            image_path = file_path
            # Display the image
            img = Image.open(file_path)
            img = img.resize((200, 200), Image.Resampling.LANCZOS)
            img_tk = ImageTk.PhotoImage(img)
            self.image_label.configure(image=img_tk, text="")
            self.image_label.image = img_tk  # Keep a reference
            print(f"Image uploaded: {file_path}")

    def start_recording(self):
        if not image_path:
            messagebox.showwarning("Warning", "Please upload an image first!")
            return

        self.record_btn.config(state=tk.DISABLED)
        self.question_label.config(text="Recording... Speak your question (10s)")
        self.root.update()

        # Start recording in a separate thread
        threading.Thread(target=self.record_audio, daemon=True).start()

    def record_audio(self):
        global is_recording
        is_recording = True

        # Record audio
        print("Recording audio...")
        audio = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
        sd.wait()  # Wait until recording is finished

        # Save audio to WAV file
        audio = (audio * 32767).astype(np.int16)  # Convert to 16-bit PCM
        wavfile.write(audio_file, SAMPLE_RATE, audio)

        print("Recording finished.")
        is_recording = False

        # Transcribe audio
        self.transcribe_audio()

    def transcribe_audio(self):
        if not os.path.exists(audio_file):
            self.question_label.config(text="Error: Audio file not found.")
            self.record_btn.config(state=tk.NORMAL)
            return

        # Transcribe using Whisper
        print("Transcribing audio...")
        result = whisper_model.transcribe(audio_file, language="en")
        question = result["text"].strip()
        print(f"Transcribed question: {question}")

        self.question_label.config(text=f"Question: {question}")
        self.record_btn.config(state=tk.NORMAL)

        # Generate answer using BLIP-2
        self.generate_answer(question)

    def generate_answer(self, question):
        # Load the image
        image = Image.open(image_path).convert('RGB')

        # Prepare inputs for BLIP-2
        inputs = blip_processor(images=image, text=question, return_tensors="pt").to(device)

        # Generate answer
        print("Generating answer...")
        outputs = blip_model.generate(
            **inputs,
            max_length=50,
            num_beams=5,
            early_stopping=True
        )
        answer = blip_processor.decode(outputs[0], skip_special_tokens=True)
        print(f"Answer: {answer}")

        # Update GUI
        self.answer_label.config(text=f"Answer: {answer}")
        self.current_answer = answer
        self.play_btn.config(state=tk.NORMAL)

        # Handle edge cases
        if not answer:
            self.answer_label.config(text="Answer: Sorry, I couldn't generate an answer.")
            self.play_btn.config(state=tk.DISABLED)

    def play_answer(self):
        if not self.current_answer:
            return

        # Play the answer using TTS in a separate thread to avoid freezing the GUI
        threading.Thread(target=self.speak_answer, daemon=True).start()

    def speak_answer(self):
        print("Speaking answer...")
        tts_engine.say(self.current_answer)
        tts_engine.runAndWait()
        print("Finished speaking.")

# Main application
if __name__ == "__main__":
    root = tk.Tk()
    app = VQAInterface(root)
    root.mainloop()

# Clean up
if os.path.exists(audio_file):
    os.remove(audio_file)